# HealthConnect Clinic - Week 4 Initial Data Analysis
**Track:** Data Analytics
**Project:** HealthConnect Clinic Experience Lab
**Central Project Question:** How can HealthConnect Clinic use data and AI to reduce missed appointments and improve the patient support experience?

This notebook covers the Week 4 deliverables for the Data Analytics track:
0. Project resources review (dataset + data dictionary)
1. Dataset overview
2. Data quality assessment (structural + logical + cross-field consistency)
3. Relevant business questions
4. Potential KPIs - identified, justified, and explored with real numbers
5. Initial analysis approach
6. Assumptions, limitations, risks & dependencies

## 0. Project Resources Review

Per the assignment, the Data Analytics track's primary resources are:
- HealthConnect_Appointment_Data.csv
- the appointment dataset
- HealthConnect_Data_Dictionary.xlsx
- variable definitions and rules

The HealthConnect_Clinic_Knowledge_Base.docx is out of scope for this track (its primary users are the Generative AI track).

Both resources are loaded and cross-checked below before any analysis begins, to confirm they are aligned.

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('../data/raw/HealthConnect_Appointment_Data.csv')
data_dict = pd.read_excel('../data/raw/HealthConnect_Data_Dictionary.xlsx')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Data dictionary entries: {len(data_dict)}")
data_dict

Dataset shape: 5000 rows, 18 columns
Data dictionary entries: 18


,Variable,Data Type,Description,Example,Notes
0,appointment_id,Text,Unique appointment identifier,HC-00001,Primary key
1,patient_id,Text,Anonymised patient identifier,P-0421,May appear across multiple appointments
2,gender,Text,Recorded gender category,Female,Synthetic and anonymised
3,age,Integer,Patient age in years,34,Adult patients only
4,age_group,Text,Age band derived from age,25-34,Provided for descriptive analysis
5,appointment_type,Text,Type of scheduled appointment,Follow-up,Four appointment categories
6,booking_date,Date,Date appointment was booked,2025-04-10 00:00:00,ISO format
7,appointment_date,Date,Scheduled appointment date,2025-04-24 00:00:00,ISO format
8,appointment_day,Text,Day of week for appointment,Thursday,Derived from appointment date
9,appointment_time,Text,Appointment time period,Morning,"Morning, Afternoon, Evening"


In [3]:
# Cross-check: every dataset column must be documented in the data dictionary, and vice versa
dd_vars = set(data_dict['Variable'].tolist())
csv_cols = set(df.columns.tolist())

print("Columns in CSV but missing from Data Dictionary:", csv_cols - dd_vars)
print("Variables in Data Dictionary but missing from CSV:", dd_vars - csv_cols)

Columns in CSV but missing from Data Dictionary: set()
Variables in Data Dictionary but missing from CSV: set()


Every column in the dataset is documented in the data dictionary, and every dictionary entry has a matching column in the dataset. The two resources are aligned, no undocumented or missing fields.

Chaque colonne du dataset est documentée dans le data dictionary, et chaque entrée du dictionnaire correspond à une colonne du dataset. Les deux ressources sont alignées, aucun champ non documenté ou manquant.

## 1. Dataset Overview

In [4]:
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object 
 15  dist

In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
appointment_id,5000,5000,HC-00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_id,5000,1696,P-1399,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,5000,3,Female,2488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,5000.0,NaN,NaN,NaN,48.7948,18.138547,18.0,33.0,49.0,64.0,80.0
age_group,5000,6,65+,1241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
appointment_type,5000,4,General Consultation,2086,NaN,NaN,NaN,NaN,NaN,NaN,NaN
booking_date,5000,593,8/24/2025,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
appointment_date,5000,545,3/14/2026,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
appointment_day,5000,7,Wednesday,737,NaN,NaN,NaN,NaN,NaN,NaN,NaN
appointment_time,5000,3,Morning,2227,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
categorical_cols = ['gender', 'age_group', 'appointment_type', 'appointment_day',
                    'appointment_time', 'reminder_sent', 'reminder_channel', 'appointment_outcome']
for col in categorical_cols:
    print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts(dropna=False))


--- gender (3 unique values) ---
gender
Female               2488
Male                 2404
Prefer not to say     108
Name: count, dtype: int64

--- age_group (6 unique values) ---
age_group
65+      1241
35-44     819
55-64     800
45-54     793
25-34     783
18-24     564
Name: count, dtype: int64

--- appointment_type (4 unique values) ---
appointment_type
General Consultation       2086
Follow-up                  1421
Specialist Consultation     900
Diagnostic Test             593
Name: count, dtype: int64

--- appointment_day (7 unique values) ---
appointment_day
Wednesday    737
Sunday       737
Friday       728
Saturday     724
Monday       701
Tuesday      689
Thursday     684
Name: count, dtype: int64

--- appointment_time (3 unique values) ---
appointment_time
Morning      2227
Afternoon    2094
Evening       679
Name: count, dtype: int64

--- reminder_sent (2 unique values) ---
reminder_sent
Yes    3634
No     1366
Name: count, dtype: int64

--- reminder_channel (3 unique v

In [8]:
print("Overall appointment_outcome distribution:")
print(df['appointment_outcome'].value_counts())
print()
print((df['appointment_outcome'].value_counts(normalize=True)*100).round(2).astype(str) + '%')

Overall appointment_outcome distribution:
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

appointment_outcome
No-Show      48.46%
Attended     46.28%
Cancelled     5.26%
Name: proportion, dtype: object


- 5,000 appointment records, 18 columns, one row per appointment.
- appointment_id is unique (primary key, confirmed later in Data Quality). patient_id repeats across rows up to 9 appointments for the same patient since patients can book multiple times.
- appointment_outcome is the key outcome variable, split into: No-Show 48.46%, Attended 46.28%, Cancelled 5.26%. Close to half of all appointments are missed, this confirms the scale of the business problem described in the scenario.
- Four appointment types (General Consultation is most common, 41.7%), three time slots (Morning most common), all seven weekdays are represented fairly evenly (approx. 680 - 740 each).
- Reminders were sent for 72.7% of appointments (3,634 of 5,000); of those, SMS is the dominant channel (2,000), followed by WhatsApp (1,101) and Email (533).
- Age ranges from 18 to 80 (mean ~48.8), consistent with the "Adult patients only" rule in the data dictionary. age_group is a derived categorical version of age.

- 5 000 enregistrements de rendez-vous, 18 colonnes, une ligne par rendez-vous.
- appointment_id est unique (clé primaire, confirmé plus loin dans la qualité des données). patient_id se répète jusqu'à 9 rendez-vous pour un même patient car les patients peuvent réserver plusieurs fois.
- appointment_outcome est la variable clé : No-Show 48,46%, Attended (présent) 46,28%, Cancelled (annulé) 5,26%. Près de la moitié des rendez-vous sont manqués, cela confirme l'ampleur du problème business décrit dans le scénario.
- Quatre types de rendez-vous (General Consultation le plus fréquent, 41,7%), trois créneaux horaires (le matin le plus fréquent), les sept jours de la semaine sont assez équilibrés (environ 680-740 chacun).
- Des rappels ont été envoyés pour 72,7% des rendez-vous (3 634 sur 5 000) ; parmi eux, le SMS est le canal dominant (2 000), suivi de WhatsApp (1 101) et de l'Email (533).
- L'âge varie de 18 à 80 ans (moyenne ~48,8), cohérent avec la règle "Adult patients only" du data dictionary. age_group est une version catégorielle dérivée de age.

## 2. Data Quality Assessment

This section goes beyond a basic missing-value check: it validates structural integrity, logical business rules, and cross-field consistency, so that no data issue is discovered later during analysis.

In [12]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
quality_df[quality_df['missing_count'] > 0].sort_values('missing_count', ascending=False)

,missing_count,missing_pct
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


In [13]:
print("Duplicate appointment_id values:", df['appointment_id'].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Duplicate appointment_id values: 0
Fully duplicated rows: 0


In [15]:
inconsistent_no_shows = df[df['previous_no_shows'] > df['previous_appointments']]
print("Rows where previous_no_shows > previous_appointments:", len(inconsistent_no_shows))

negative_lead = df[df['booking_lead_days'] < 0]
print("Rows with negative booking_lead_days:", len(negative_lead))

mismatch_reminder = df[(df['reminder_sent'] == 'Yes') & (df['reminder_channel'].isna())]
mismatch_reminder2 = df[(df['reminder_sent'] == 'No') & (df['reminder_channel'].notna())]
print("Rows: reminder_sent = Yes but reminder_channel missing:", len(mismatch_reminder))
print("Rows: reminder_sent = No but reminder_channel filled:", len(mismatch_reminder2))

print("Rows with age < 18:", len(df[df['age'] < 18]))
print("Age range:", df['age'].min(), "-", df['age'].max())

Rows where previous_no_shows > previous_appointments: 0
Rows with negative booking_lead_days: 0
Rows: reminder_sent = Yes but reminder_channel missing: 0
Rows: reminder_sent = No but reminder_channel filled: 0
Rows with age < 18: 0
Age range: 18 - 80


In [16]:
def expected_group(age):
    if age <= 24: return '18-24'
    elif age <= 34: return '25-34'
    elif age <= 44: return '35-44'
    elif age <= 54: return '45-54'
    elif age <= 64: return '55-64'
    else: return '65+'

df['expected_age_group'] = df['age'].apply(expected_group)
mismatch_age_group = df[df['age_group'] != df['expected_age_group']]
print("Rows where age_group doesn't match derived value from age:", len(mismatch_age_group))

df['booking_date_parsed'] = pd.to_datetime(df['booking_date'], format='mixed', errors='coerce')
df['appointment_date_parsed'] = pd.to_datetime(df['appointment_date'], format='mixed', errors='coerce')
df['actual_weekday'] = df['appointment_date_parsed'].dt.day_name()
mismatch_day = df[df['appointment_day'] != df['actual_weekday']]
print("Rows where appointment_day doesn't match actual weekday of appointment_date:", len(mismatch_day))

df['computed_lead_days'] = (df['appointment_date_parsed'] - df['booking_date_parsed']).dt.days
mismatch_lead = df[df['computed_lead_days'] != df['booking_lead_days']]
print("Rows where booking_lead_days doesn't match computed value:", len(mismatch_lead))

bad_dates = df[df['appointment_date_parsed'] < df['booking_date_parsed']]
print("Rows where appointment_date is before booking_date:", len(bad_dates))

unparseable = df[df['booking_date_parsed'].isna() | df['appointment_date_parsed'].isna()]
print("Rows with unparseable dates:", len(unparseable))

Rows where age_group doesn't match derived value from age: 0
Rows where appointment_day doesn't match actual weekday of appointment_date: 0
Rows where booking_lead_days doesn't match computed value: 0
Rows where appointment_date is before booking_date: 0
Rows with unparseable dates: 0


In [17]:
dupe_bookings = df[df.duplicated(subset=['patient_id', 'appointment_date', 'appointment_time'], keep=False)]
dupe_bookings_sorted = dupe_bookings.sort_values(['patient_id', 'appointment_date', 'appointment_time'])
print(f"Rows involved in duplicate patient/date/time-slot bookings: {len(dupe_bookings)}")
print(f"Distinct patient/date/time-slot groups affected: {dupe_bookings.groupby(['patient_id','appointment_date','appointment_time']).ngroups}")
dupe_bookings_sorted[['appointment_id','patient_id','appointment_date','appointment_time','appointment_type','appointment_outcome']]

Rows involved in duplicate patient/date/time-slot bookings: 18
Distinct patient/date/time-slot groups affected: 9


,appointment_id,patient_id,appointment_date,appointment_time,appointment_type,appointment_outcome
1357,HC-01358,P-0039,3/15/2025,Morning,Diagnostic Test,Attended
3307,HC-03308,P-0039,3/15/2025,Morning,General Consultation,No-Show
2840,HC-02841,P-0273,8/28/2025,Morning,Specialist Consultation,Attended
3080,HC-03081,P-0273,8/28/2025,Morning,Specialist Consultation,No-Show
222,HC-00223,P-0391,1/20/2026,Morning,General Consultation,No-Show
4566,HC-04567,P-0391,1/20/2026,Morning,General Consultation,No-Show
3248,HC-03249,P-0575,3/18/2025,Afternoon,Follow-up,Attended
4025,HC-04026,P-0575,3/18/2025,Afternoon,Specialist Consultation,No-Show
3730,HC-03731,P-1171,1/30/2026,Morning,General Consultation,No-Show
4945,HC-04946,P-1171,1/30/2026,Morning,General Consultation,Attended


In [18]:
df[['age', 'booking_lead_days', 'previous_appointments', 'previous_no_shows',
    'distance_to_clinic_km', 'waiting_time_minutes']].describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


- Missing values are limited to three fields: reminder_channel (27.32% expected, since it is only populated when a reminder was sent), distance_to_clinic_km (1.80%) and waiting_time_minutes (1.20%). This matches the data dictionary's note of "limited missing values" for the last two.
- No duplicate appointment_id values and no fully duplicated rows, the primary key is clean.
- All logical business rules hold: previous_no_shows never exceeds previous_appointments, booking_lead_days is never negative, reminder_channel is populated if and only if reminder_sent = Yes, and all patients are adults (18–80).
- All derived fields are internally consistent: age_group always matches the value derived from age, appointment_day always matches the actual weekday of appointment_date, and booking_lead_days always equals the exact day difference between booking_date and appointment_date. No date ordering issues and no unparseable dates.
- One genuine anomaly found: 18 rows (9 patient/date/time-slot groups) show the same patient with two different appointment IDs on the same date and same time slot, sometimes with different appointment types and different outcomes (e.g., patient P-0039 has one "Attended" Diagnostic Test and one "No-Show" General Consultation, both on 3/15/2025 in the Morning). This could reflect either genuine double-booking of two different services at the same time, or a data-entry/generation duplication. It should be flagged and clarified before being used to train any predictive model (Data Science track), since it could introduce noise or record-level double-counting.

- Les valeurs manquantes se limitent à trois champs : reminder_channel (27,32% normal, car rempli uniquement quand un rappel a été envoyé), distance_to_clinic_km (1,80%) et waiting_time_minutes (1,20%). Cela correspond à la note du data dictionary sur des "valeurs manquantes limitées" pour ces deux derniers champs.
- Aucun doublon sur appointment_id et aucune ligne entièrement dupliquée, la clé primaire est propre.
- Toutes les règles métier logiques sont respectées : previous_no_shows ne dépasse jamais previous_appointments, booking_lead_days n'est jamais négatif, reminder_channel est rempli si et seulement si reminder_sent = Yes, et tous les patients sont des adultes (18-80 ans).
- Tous les champs dérivés sont cohérents en interne : age_group correspond toujours à la valeur dérivée de age, appointment_day correspond toujours au jour réel de la semaine de appointment_date, et booking_lead_days est toujours égal à l'écart exact de jours entre booking_date et appointment_date. Aucun problème d'ordre de dates, aucune date illisible.
- Une anomalie réelle a été détectée : 18 lignes (9 groupes patient/date/créneau) montrent le même patient avec deux appointment_id différents à la même date et au même créneau horaire, parfois avec des types de rendez-vous et des issues différentes (ex. le patient P-0039 a un Diagnostic Test "Attended" et une General Consultation "No-Show", tous deux le 15/03/2025 le matin). Cela peut refléter soit un double rendez-vous légitime pour deux services distincts au même moment, soit une duplication liée à la génération/saisie des données. Ce point doit être signalé et clarifié avant d'être utilisé pour entraîner un modèle prédictif (piste Data Science), car cela pourrait introduire du bruit ou un double comptage au niveau des enregistrements.

## 3. Relevant Business Questions

1. What is the overall no-show rate, and how does it vary across appointment types, time slots, and days of the week?
2. Which patient or booking characteristics (age group, distance to clinic, booking lead time, previous no-show history) are most associated with missed appointments?
3. Does sending a reminder and which channel (SMS, WhatsApp, Email) correlate with a lower no-show rate?
4. How much appointment capacity is lost to no-shows and cancellations combined, and does this vary by appointment type or time of day?
5. Do longer waiting times or booking lead times correlate with a higher likelihood of no-show or cancellation?

1. Quel est le taux global de no-show, et comment varie-t-il selon le type de rendez-vous, le créneau horaire et le jour de la semaine ?
2. Quelles caractéristiques du patient ou de la réservation (tranche d'âge, distance à la clinique, délai de réservation, historique de no-shows) sont les plus associées aux rendez-vous manqués ?
3. L'envoi d'un rappel et quel canal (SMS, WhatsApp, Email)  est-il corrélé à un taux de no-show plus faible ?
4. Quelle proportion de capacité de rendez-vous est perdue à cause des no-shows et des annulations combinés, et cela varie-t-il selon le type de rendez-vous ou le moment de la journée ?
5. Des temps d'attente ou des délais de réservation plus longs sont-ils corrélés à une probabilité plus élevée de no-show ou d'annulation ?

## 4. Potential KPIs - Identification, Justification & Initial Exploration

The assignment only requires KPIs to be identified and justified at this stage, not calculated or visualised. However, to fully understand the data before committing to KPIs, each one is explored below with real figures, this ensures no gap is discovered later and that each KPI is genuinely grounded in the data.

In [19]:
print("Overall outcome rate:")
print((df['appointment_outcome'].value_counts(normalize=True)*100).round(2))

print("\nBy appointment_type:")
print((pd.crosstab(df['appointment_type'], df['appointment_outcome'], normalize='index')*100).round(2))

print("\nBy appointment_time:")
print((pd.crosstab(df['appointment_time'], df['appointment_outcome'], normalize='index')*100).round(2))

print("\nBy appointment_day:")
print((pd.crosstab(df['appointment_day'], df['appointment_outcome'], normalize='index')*100).round(2))

Overall outcome rate:
appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64

By appointment_type:
appointment_outcome      Attended  Cancelled  No-Show
appointment_type                                     
Diagnostic Test             45.87       4.38    49.75
Follow-up                   43.35       5.42    51.23
General Consultation        48.27       5.08    46.64
Specialist Consultation     46.56       6.00    47.44

By appointment_time:
appointment_outcome  Attended  Cancelled  No-Show
appointment_time                                 
Afternoon               46.13       5.49    48.38
Evening                 44.77       5.45    49.78
Morning                 46.88       4.98    48.14

By appointment_day:
appointment_outcome  Attended  Cancelled  No-Show
appointment_day                                  
Friday                  49.04       4.40    46.57
Monday                  43.79       6.56    49.64
Saturday                47.51

In [20]:
print("By reminder_sent:")
print((pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index')*100).round(2))

df['reminder_channel_filled'] = df['reminder_channel'].fillna('No Reminder')
print("\nBy reminder_channel (including 'No Reminder'):")
print((pd.crosstab(df['reminder_channel_filled'], df['appointment_outcome'], normalize='index')*100).round(2))

By reminder_sent:
appointment_outcome  Attended  Cancelled  No-Show
reminder_sent                                    
No                      42.68       5.93    51.39
Yes                     47.63       5.01    47.36

By reminder_channel (including 'No Reminder'):
appointment_outcome      Attended  Cancelled  No-Show
reminder_channel_filled                              
Email                       46.53       5.07    48.41
No Reminder                 42.68       5.93    51.39
SMS                         49.60       4.65    45.75
WhatsApp                    44.60       5.63    49.77


In [21]:
had_prev_noshow = df[df['previous_no_shows'] > 0]
no_prev_noshow = df[df['previous_no_shows'] == 0]

print("Outcome distribution | patients WITH previous no-shows:")
print((had_prev_noshow['appointment_outcome'].value_counts(normalize=True)*100).round(2))

print("\nOutcome distribution | patients WITHOUT previous no-shows:")
print((no_prev_noshow['appointment_outcome'].value_counts(normalize=True)*100).round(2))

Outcome distribution | patients WITH previous no-shows:
appointment_outcome
No-Show      55.41
Attended     40.40
Cancelled     4.18
Name: proportion, dtype: float64

Outcome distribution | patients WITHOUT previous no-shows:
appointment_outcome
Attended     50.46
No-Show      43.51
Cancelled     6.03
Name: proportion, dtype: float64


In [22]:
lost_overall = df['appointment_outcome'].isin(['No-Show', 'Cancelled']).mean() * 100
print(f"Overall lost capacity rate: {lost_overall:.2f}%")

print("\nBy appointment_type:")
print(df.groupby('appointment_type')['appointment_outcome'].apply(lambda x: x.isin(['No-Show','Cancelled']).mean()*100).round(2))

print("\nBy appointment_time:")
print(df.groupby('appointment_time')['appointment_outcome'].apply(lambda x: x.isin(['No-Show','Cancelled']).mean()*100).round(2))

Overall lost capacity rate: 53.72%

By appointment_type:
appointment_type
Diagnostic Test            54.13
Follow-up                  56.65
General Consultation       51.73
Specialist Consultation    53.44
Name: appointment_outcome, dtype: float64

By appointment_time:
appointment_time
Afternoon    53.87
Evening      55.23
Morning      53.12
Name: appointment_outcome, dtype: float64


In [23]:
print("Average distance_to_clinic_km by outcome:")
print(df.groupby('appointment_outcome')['distance_to_clinic_km'].mean().round(2))

print("\nAverage waiting_time_minutes by outcome:")
print(df.groupby('appointment_outcome')['waiting_time_minutes'].mean().round(2))

print("\nAverage booking_lead_days by outcome:")
print(df.groupby('appointment_outcome')['booking_lead_days'].mean().round(2))

print("\nNo-show rate by age_group:")
print((pd.crosstab(df['age_group'], df['appointment_outcome'], normalize='index')*100).round(2))

Average distance_to_clinic_km by outcome:
appointment_outcome
Attended      9.67
Cancelled    10.11
No-Show      10.53
Name: distance_to_clinic_km, dtype: float64

Average waiting_time_minutes by outcome:
appointment_outcome
Attended     24.29
Cancelled    23.21
No-Show      24.20
Name: waiting_time_minutes, dtype: float64

Average booking_lead_days by outcome:
appointment_outcome
Attended     24.52
Cancelled    29.63
No-Show      34.53
Name: booking_lead_days, dtype: float64

No-show rate by age_group:
appointment_outcome  Attended  Cancelled  No-Show
age_group                                        
18-24                   45.57       4.26    50.18
25-34                   45.08       4.21    50.70
35-44                   45.67       5.98    48.35
45-54                   46.03       5.93    48.05
55-64                   45.00       4.25    50.75
65+                     48.75       6.12    45.12


| # | KPI | Linked Question | What the exploration shows |
|---|-----|------------------|------------------------------|
| 1 | No-Show Rate (%) | Q1 | Overall rate is 48.46%, nearly one in two appointments. Varies modestly by type (Follow-up highest at 51.2%, General Consultation lowest at 46.6%) and by day (Sunday 50.5% vs. Friday 46.6%), but the spread is narrow (about 5 points): no single segment stands out dramatically. |
| 2 | No-Show Rate by Reminder Status/Channel | Q3 | Patients with no reminder no-show at 51.4% vs. 47.4% when a reminder was sent, a real but moderate effect (about 4 points). SMS performs best (45.75% no-show) vs. WhatsApp (49.77%) and Email (48.41%). This is a genuinely actionable signal worth deeper testing. |
| 3 | Repeat No-Show Rate | Q2 | Patients with a history of no-shows no-show again at 55.4%, vs. 43.5% for those with no prior no-shows: an 11.9-point gap, the strongest signal found in this exploration. Past behaviour is clearly predictive, which is directly useful for the Data Science track's model. |
| 4 | Lost Appointment Capacity Rate | Q4 | 53.72% of all appointments end in No-Show or Cancelled, over half of scheduled capacity is not realised as planned. This is consistently high across appointment types (51.7% to 56.7%) and time slots (53.1% to 55.2%), indicating a systemic rather than segment-specific issue. |
| 5 | Distance / Waiting Time / Lead Time by Outcome | Q2, Q5 | Distance and waiting time show almost no difference between outcomes: weak signals. Booking lead time is more notable: No-Shows book 34.5 days ahead on average vs. 24.5 days for Attended, a 10-day gap, suggesting longer lead times may weaken commitment. |

Overall takeaway: the strongest predictive/actionable signals found are (1) prior no-show history, (2) reminder status/channel, and (3) booking lead time. Distance and waiting time appear to have little standalone relationship with attendance in this dataset. This should be validated further (e.g. with correlation coefficients or a simple model) in the next phase, not used as a final conclusion.

| # | KPI | Question liée | Ce que l'exploration montre |
|---|-----|-----------------|-------------------------------|
| 1 | Taux de No-Show (%) | Q1 | Le taux global est de 48,46%, près d'un rendez-vous sur deux. Il varie modestement selon le type (Follow-up le plus élevé à 51,2%, General Consultation le plus bas à 46,6%) et selon le jour (dimanche 50,5% vs. vendredi 46,6%), mais l'écart reste faible (environ 5 points) : aucun segment ne se démarque fortement. |
| 2 | Taux de No-Show par statut/canal de rappel | Q3 | Les patients sans rappel font un no-show à 51,4% contre 47,4% quand un rappel a été envoyé, un effet réel mais modéré (environ 4 points). Le SMS performe le mieux (45,75% de no-show) devant WhatsApp (49,77%) et l'Email (48,41%). C'est un signal réellement actionnable, à approfondir. |
| 3 | Taux de récidive de No-Show | Q2 | Les patients ayant un historique de no-shows refont un no-show à 55,4%, contre 43,5% pour ceux sans antécédent : un écart de 11,9 points, le signal le plus fort trouvé dans cette exploration. Le comportement passé est clairement prédictif, ce qui est directement utile pour le modèle de la piste Data Science. |
| 4 | Taux de capacité perdue | Q4 | 53,72% de tous les rendez-vous se terminent en No-Show ou Annulation, soit plus de la moitié de la capacité planifiée non réalisée. Ce taux reste élevé de façon homogène selon les types de rendez-vous (51,7% à 56,7%) et les créneaux (53,1% à 55,2%), ce qui indique un problème systémique plutôt que localisé à un segment. |
| 5 | Distance / Temps d'attente / Délai de réservation par issue | Q2, Q5 | La distance et le temps d'attente montrent presque aucune différence entre les issues : signaux faibles. Le délai de réservation est plus notable : les No-Shows réservent en moyenne 34,5 jours à l'avance contre 24,5 jours pour les Attended, un écart de 10 jours, suggérant qu'un délai plus long pourrait affaiblir l'engagement du patient. |

Conclusion générale : les signaux prédictifs/actionnables les plus forts sont (1) l'historique de no-show antérieur, (2) le statut/canal de rappel, et (3) le délai de réservation. La distance et le temps d'attente semblent avoir peu de lien direct avec la présence dans ce dataset. Ce point devra être validé plus rigoureusement (ex. coefficients de corrélation ou modèle simple) lors de la phase suivante, et ne doit pas être considéré comme une conclusion définitive à ce stade.

## 5. Initial Analysis Approach

1. Prioritise the three strongest signals for deeper statistical validation: previous no-show history, reminder status/channel, and booking lead time.
2. Quantify formally: compute correlation coefficients or run a simple logistic regression to confirm which factors are statistically significant, rather than relying on descriptive cross-tabs alone.
3. Segment-level dashboards: visualise no-show and lost-capacity rates by appointment type, day, and time slot to support operational decisions (e.g., overbooking policy for high-risk slots).
4. Investigate the duplicate-booking anomaly (Section 2, 9 cases) with the clinic/data source before drawing firm conclusions from records involving those patients.
5. Align with the Data Science track: share these findings so the no-show prediction model is built on the same validated signals, avoiding duplicated exploration.
6. Document and iterate: update this analysis as new data or clarifications become available.

1. Prioriser les trois signaux les plus forts pour une validation statistique plus poussée : historique de no-show antérieur, statut/canal de rappel, et délai de réservation.
2. Quantifier formellement : calculer des coefficients de corrélation ou exécuter une régression logistique simple pour confirmer quels facteurs sont statistiquement significatifs, plutôt que de se fier uniquement aux tableaux croisés descriptifs.
3. Tableaux de bord par segment : visualiser les taux de no-show et de capacité perdue par type de rendez-vous, jour et créneau horaire, pour appuyer des décisions opérationnelles (ex. politique de surbooking sur les créneaux à risque).
4. Investiguer l'anomalie de double réservation (Section 2, 9 cas) avec la clinique/source de données avant de tirer des conclusions fermes sur les enregistrements concernés.
5. S'aligner avec la piste Data Science : partager ces résultats pour que le modèle de prédiction de no-show s'appuie sur les mêmes signaux validés, en évitant une exploration redondante.
6. Documenter et itérer : mettre à jour cette analyse au fur et à mesure que de nouvelles données ou clarifications sont disponibles.

## 6. Assumptions, Limitations, Risks & Dependencies

- Synthetic data: the dataset is fictional/anonymised; patterns found may not generalise to a real clinic.
- No cost/revenue data: cannot yet quantify the financial impact of no-shows, only counts and rates.
- Missing values in reminder_channel, distance_to_clinic_km, waiting_time_minutes are limited but should be handled explicitly (exclude vs. impute) before deeper statistical modelling.
- Duplicate-booking anomaly (9 patient/date/time-slot groups, 18 rows): unresolved. Could indicate legitimate double-service bookings or a data artefact. Flagged as a risk for any downstream modelling that assumes one appointment per patient per slot.
- Cancelled vs. No-Show are distinct outcomes with likely different causes (a cancellation is a deliberate patient action, a no-show is not). Combining them into a single "lost capacity" KPI is useful operationally but should not obscure this causal difference in later analysis.
- Correlational, not causal, findings: all relationships described in Section 4 are associations observed in cross-tabs. No statistical significance testing or control for confounding variables has been performed yet.
- Dependency on Data Science track: this analysis lays groundwork the ML track will build on for no-show prediction. Findings should be shared and aligned to avoid duplicated exploration.
- Time constraint: Week 4 is scoped to understanding/planning; formal visualisation and statistical testing are deferred to later weeks, per the assignment's explicit scope.

- Données synthétiques : le dataset est fictif/anonymisé ; les tendances observées peuvent ne pas se généraliser à une vraie clinique.
- Absence de données de coût/revenu : impossible pour l'instant de quantifier l'impact financier des no-shows, seulement les comptages et taux.
- Valeurs manquantes dans reminder_channel, distance_to_clinic_km, waiting_time_minutes : limitées mais à traiter explicitement (exclusion vs. imputation) avant une modélisation statistique plus poussée.
- Anomalie de double réservation (9 groupes patient/date/créneau, 18 lignes) : non résolue. Pourrait indiquer des réservations légitimes pour deux services distincts, ou un artefact de génération des données. Signalée comme un risque pour toute modélisation en aval qui suppose un seul rendez-vous par patient et par créneau.
- Annulé vs. No-Show sont des issues distinctes avec probablement des causes différentes (une annulation est une action délibérée du patient, un no-show ne l'est pas). Les combiner dans un seul KPI de "capacité perdue" est utile opérationnellement mais ne doit pas masquer cette différence de nature dans l'analyse ultérieure.
- Résultats corrélationnels, pas causaux : toutes les relations décrites en Section 4 sont des associations observées dans des tableaux croisés. Aucun test de significativité statistique ni contrôle des facteurs de confusion n'a encore été réalisé.
- Dépendance avec la piste Data Science : cette analyse pose les bases sur lesquelles la piste ML s'appuiera pour la prédiction des no-shows. Les résultats doivent être partagés et alignés pour éviter une exploration redondante.
- Contrainte de temps : la Semaine 4 se limite à la compréhension/planification ; la visualisation formelle et les tests statistiques sont reportés aux semaines suivantes, conformément au périmètre explicite de la consigne.